# Multi-Stock LSTM Training Pipeline

This notebook trains a **separate LSTM model** for each of the following stocks:

| # | Ticker | Description |
|---|--------|-------------|
| 1 | `META` | Meta Platforms |
| 2 | `MSFT` | Microsoft |
| 3 | `GC=F` | Gold Futures |
| 4 | `JUFU.TO` | Jushi Holdings (Canadian exchange) |
| 5 | `SWEDY` | Schneider Electric ADR (closest available for SWEDY/Cairo) |
| 6 | `HPQ` | HP Inc. |
| 7 | `NVDA` | NVIDIA Corporation |

> **Note on JUFU.CA / SWEDY:** `JUFU.CA` is mapped to `JUFU.TO` (Toronto Stock Exchange) since yfinance uses `.TO` suffix for Canadian stocks. `SWEDY` (Schneider Electric ADR) is the closest publicly-tradeable proxy for the Schneider Electric Cairo listing — the Cairo Stock Exchange is not supported by yfinance.

Each stock gets:
- Its own dataset with feature engineering
- Its own trained LSTM model
- Its own loss curve visualization
- Its own Actual vs Predicted visualization
- Its own evaluation metrics (RMSE, MAE, R2, Directional Accuracy)

In [ ]:
# ============================================================
# CELL 1: Imports
# ============================================================
import torch
import numpy as np
import pandas as pd
import yfinance as yf
import ta
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from tqdm import tqdm

print("All libraries imported successfully.")
print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# ============================================================
# CELL 2: Stock Configuration
# ============================================================

STOCKS = [
    {"ticker": "META",    "name": "Meta Platforms"},
    {"ticker": "MSFT",    "name": "Microsoft"},
    {"ticker": "GC=F",    "name": "Gold Futures"},
    {"ticker": "JUFU.TO", "name": "Jushi Holdings (TSX)"},
    {"ticker": "SWEDY",   "name": "Schneider Electric ADR"},
    {"ticker": "HPQ",     "name": "HP Inc."},
    {"ticker": "NVDA",    "name": "NVIDIA"},
]

# Hyperparameters (same for all models — identical to original notebook)
WINDOW_SIZE  = 60
TEST_SIZE    = 0.1
HIDDEN_SIZE  = 32
NUM_LAYERS   = 1
BATCH_SIZE   = 32
EPOCHS       = 100
LEARNING_RATE = 4e-3
PERIOD        = "20y"

FEATURES = ['Close', 'Day_of_Week', 'Month', 'Volume', 'return', 'ma20', 'ma50', 'volatility', 'rsi']

print(f"Will train {len(STOCKS)} separate LSTM models.")
for i, s in enumerate(STOCKS, 1):
    print(f"  {i}. [{s['ticker']}] {s['name']}")

In [ ]:
# ============================================================
# CELL 3: Helper Functions
# (Same logic as original notebook, wrapped into reusable functions)
# ============================================================

# ---- Dataset & Model Classes ----

class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc   = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        out = out[:, -1, :]
        out = self.fc(out)
        return out


# ---- Data Pipeline ----

def prepare_dataset(ticker, period="20y"):
    """Download, feature-engineer, scale and sequence data for one ticker."""
    print(f"\n[{ticker}] Downloading data...")
    df = yf.Ticker(ticker).history(period=period)
    df = df.reset_index()
    df.dropna(inplace=True)

    if len(df) < WINDOW_SIZE + 60:
        raise ValueError(f"[{ticker}] Not enough data ({len(df)} rows). Skipping.")

    # Feature engineering — identical to original notebook
    df['Day_of_Week'] = df['Date'].dt.dayofweek
    df['Month']       = df['Date'].dt.month
    df['Day_name']    = df['Date'].dt.day_name()

    df['return']     = df['Close'].pct_change()
    df['ma20']       = df['Close'].rolling(window=20).mean()
    df['ma50']       = df['Close'].rolling(window=50).mean()
    df['volatility'] = df['return'].rolling(window=20).std() * np.sqrt(252)
    df['rsi']        = ta.momentum.RSIIndicator(df['Close']).rsi()

    df.dropna(inplace=True)

    # Target: tomorrow's close
    df['Target'] = df['Close'].shift(-1)
    df_model = df.dropna(subset=['Target'])

    data = df_model[FEATURES].values

    scaler        = MinMaxScaler(feature_range=(0, 1))
    scaled_data   = scaler.fit_transform(data)

    target_scaler  = MinMaxScaler(feature_range=(0, 1))
    scaled_target  = target_scaler.fit_transform(df_model[['Target']].values)

    # Sliding window sequences
    X, y = [], []
    for i in range(WINDOW_SIZE, len(scaled_data)):
        X.append(scaled_data[i - WINDOW_SIZE:i])
        y.append(scaled_target[i])
    X, y = np.array(X), np.array(y)

    # Train / test split
    split_idx = int((1 - TEST_SIZE) * len(X))
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]

    train_loader = DataLoader(StockDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(StockDataset(X_test,  y_test),  batch_size=BATCH_SIZE, shuffle=False)

    print(f"[{ticker}] Total sequences: {len(X)} | Train: {len(X_train)} | Test: {len(X_test)}")
    return train_loader, test_loader, target_scaler, X_train.shape[2]


# ---- Training Loop ----

def train_model(model, train_loader, test_loader, ticker):
    """Train the LSTM model — identical loop to original notebook."""
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    train_losses, val_losses = [], []

    for epoch in range(EPOCHS):
        model.train()
        running_loss = 0.0
        progress_bar = tqdm(train_loader, desc=f"[{ticker}] Epoch {epoch+1}/{EPOCHS}", unit="batch")

        for batch_X, batch_y in progress_bar:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            progress_bar.set_postfix(loss=loss.item())

        epoch_loss = running_loss / len(train_loader)
        train_losses.append(epoch_loss)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for val_X, val_y in test_loader:
                val_X, val_y = val_X.to(device), val_y.to(device)
                val_outputs  = model(val_X)
                v_loss = criterion(val_outputs, val_y)
                val_loss += v_loss.item()

        avg_val_loss = val_loss / len(test_loader)
        val_losses.append(avg_val_loss)

        print(f"[{ticker}] End of Epoch {epoch+1} - Train Loss: {epoch_loss:.6f} - Val Loss: {avg_val_loss:.6f}")

    return train_losses, val_losses


# ---- Visualization: Loss Curve ----

def plot_loss_curve(train_losses, val_losses, ticker, name):
    """Identical to original notebook's loss visualization."""
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        y=train_losses,
        mode='lines+markers',
        name='Training Loss'
    ))
    fig.add_trace(go.Scatter(
        y=val_losses,
        mode='lines+markers',
        name='Validation Loss',
        line=dict(dash='dash')
    ))
    fig.update_layout(
        title=f'[{ticker}] {name} — Model Loss Progress',
        xaxis_title='Epochs',
        yaxis_title='Loss (MSE)',
        template='plotly_dark',
        hovermode='x unified',
        legend=dict(x=0.01, y=0.99, bgcolor='rgba(0,0,0,0)')
    )
    fig.update_xaxes(showgrid=True, gridwidth=1)
    fig.update_yaxes(showgrid=True, gridwidth=1)
    fig.show()


# ---- Visualization: Actual vs Predicted ----

def plot_predictions(model, test_loader, target_scaler, ticker, name):
    """Identical to original notebook's prediction visualization."""
    model.eval()
    test_predictions, actual_values = [], []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X = batch_X.to(device)
            outputs = model(batch_X)
            test_predictions.append(outputs.cpu().numpy())
            actual_values.append(batch_y.numpy())

    test_predictions = np.concatenate(test_predictions, axis=0)
    actual_values    = np.concatenate(actual_values,    axis=0)

    rescaled_preds   = target_scaler.inverse_transform(test_predictions)
    rescaled_actuals = target_scaler.inverse_transform(actual_values.reshape(-1, 1))

    rescaled_preds_flat   = rescaled_preds.flatten()
    rescaled_actuals_flat = rescaled_actuals.flatten()

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        y=rescaled_actuals_flat,
        mode='lines',
        name='Actual Price'
    ))
    fig.add_trace(go.Scatter(
        y=rescaled_preds_flat,
        mode='lines',
        name='Predicted Price',
        line=dict(dash='dash')
    ))
    fig.update_layout(
        title=f'[{ticker}] {name} — Actual vs Predicted Stock Prices',
        xaxis_title='Time (Test Data Points)',
        yaxis_title='Price',
        template='plotly_dark',
        hovermode='x unified'
    )
    fig.show()

    return rescaled_actuals_flat, rescaled_preds_flat


# ---- Evaluation Metrics ----

def evaluate_financial_model(model, train_loader, test_loader, target_scaler, ticker):
    """Identical evaluation function from original notebook."""
    def get_preds(loader):
        model.eval()
        all_preds, all_actuals = [], []
        with torch.no_grad():
            for batch_X, batch_y in loader:
                batch_X = batch_X.to(device)
                outputs = model(batch_X)
                all_preds.append(outputs.cpu().numpy())
                all_actuals.append(batch_y.numpy())
        preds   = target_scaler.inverse_transform(
            np.concatenate(all_preds,   axis=0).reshape(-1, 1)).flatten()
        actuals = target_scaler.inverse_transform(
            np.concatenate(all_actuals, axis=0).reshape(-1, 1)).flatten()
        return preds, actuals

    for set_name, loader in [("Train", train_loader), ("Test", test_loader)]:
        preds, actuals = get_preds(loader)
        mse  = mean_squared_error(actuals, preds)
        rmse = np.sqrt(mse)
        mae  = mean_absolute_error(actuals, preds)
        r2   = r2_score(actuals, preds)
        acc  = (np.sign(preds) == np.sign(actuals)).sum() / len(actuals)
        print(f"--- [{ticker}] {set_name} Evaluation ---")
        print(f"RMSE:                 {rmse:.6f}")
        print(f"MAE:                  {mae:.6f}")
        print(f"R2 Score:             {r2:.6f}")
        print(f"Directional Accuracy: {acc:.2%}")
        print("-" * 35)


print("Helper functions defined. Ready to train.")

In [ ]:
# ============================================================
# CELL 4: Main Training Loop — One model per stock
# ============================================================

# Dictionary to store results for every stock
all_results = {}

for stock in STOCKS:
    ticker = stock["ticker"]
    name   = stock["name"]

    print("\n" + "=" * 60)
    print(f"  STOCK: {ticker} — {name}")
    print("=" * 60)

    # ---- Step 1: Prepare Dataset ----
    try:
        train_loader, test_loader, target_scaler, input_size = prepare_dataset(
            ticker, period=PERIOD
        )
    except Exception as e:
        print(f"[{ticker}] ERROR during data preparation: {e}")
        all_results[ticker] = None
        continue

    # ---- Step 2: Build Model ----
    model = LSTMModel(
        input_size=input_size,
        hidden_size=HIDDEN_SIZE,
        num_layers=NUM_LAYERS,
        output_size=1
    ).to(device)

    print(f"[{ticker}] Model built. Input size: {input_size}, Hidden: {HIDDEN_SIZE}, Layers: {NUM_LAYERS}")

    # ---- Step 3: Train ----
    train_losses, val_losses = train_model(model, train_loader, test_loader, ticker)

    # ---- Step 4: Visualize Loss Curve ----
    plot_loss_curve(train_losses, val_losses, ticker, name)

    # ---- Step 5: Visualize Predictions ----
    actuals, preds = plot_predictions(model, test_loader, target_scaler, ticker, name)

    # ---- Step 6: Evaluate ----
    evaluate_financial_model(model, train_loader, test_loader, target_scaler, ticker)

    # ---- Store results ----
    all_results[ticker] = {
        "model":         model,
        "train_losses":  train_losses,
        "val_losses":    val_losses,
        "target_scaler": target_scaler,
        "train_loader":  train_loader,
        "test_loader":   test_loader,
    }

    print(f"\n[{ticker}] ✅ Done!\n")

print("\n" + "=" * 60)
print(f"  ALL {len(STOCKS)} MODELS TRAINED SUCCESSFULLY")
print("=" * 60)

In [ ]:
# ============================================================
# CELL 5: Summary — Final Val R2 scores across all stocks
# ============================================================

print("\n{'='*60}")
print("  SUMMARY: Final Validation Loss per Stock")
print("{'='*60}")
print(f"{'Ticker':<12} {'Stock Name':<30} {'Final Train Loss':>18} {'Final Val Loss':>16}")
print("-" * 80)

for stock in STOCKS:
    ticker = stock["ticker"]
    name   = stock["name"]
    res = all_results.get(ticker)
    if res is None:
        print(f"{ticker:<12} {name:<30} {'FAILED':>18} {'FAILED':>16}")
    else:
        tl = res['train_losses'][-1]
        vl = res['val_losses'][-1]
        print(f"{ticker:<12} {name:<30} {tl:>18.6f} {vl:>16.6f}")

print("-" * 80)

In [ ]:
# ============================================================
# CELL 6 (Optional): Save all trained models to disk
# ============================================================

import os
os.makedirs("saved_models", exist_ok=True)

for stock in STOCKS:
    ticker = stock["ticker"]
    res = all_results.get(ticker)
    if res is not None:
        safe_ticker = ticker.replace("=", "").replace(".", "_")
        path = f"saved_models/{safe_ticker}_lstm.pt"
        torch.save(res["model"].state_dict(), path)
        print(f"[{ticker}] Model saved to: {path}")

print("\nAll models saved.")